# TTS Dataset Generation

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/OpenVoiceOS/ngi0-notebooks/blob/dev/notebooks/tts_dataset_gen.ipynb) [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenVoiceOS/ngi0-notebooks/blob/dev/notebooks/tts_dataset_gen.ipynb)

Makes a speech dataset with one voice, ready to train a text-to-speech voice from.

## Start here

1. The notebook picks 2,000 sentences from [Tatoeba](https://tatoeba.org), and the speech model [OmniVoice](https://github.com/k2-fsa/OmniVoice) reads them aloud in one voice. Tatoeba sentences are not filtered, and some contain swearing or violence.
2. You get a folder of audio clips with their text. [`train_vits.ipynb`](train_vits.ipynb) trains a small, fast voice from it.
3. Turn on **Internet** and a **GPU** (Kaggle: Settings panel, Internet on, Accelerator GPU T4), then press **Run All**. Nothing needs editing.
4. A run takes about 40 minutes on a Kaggle T4 GPU and makes about 75 minutes of speech.

| Field | Value |
|---|---|
| Inputs | Nothing by default. Optional: a language, your own voice recording, your own sentences. |
| Outputs | The folder `tts_dataset_run_<RUN_ID>`: `wavs/`, `metadata.csv`, `dataset.json`, `voice_ref.wav`, `voice_ref.txt`, `manifest.jsonl`. |
| Runtime | A measured run on a Kaggle T4 took 37 minutes for 2,000 sentences, about one clip per second. |
| Hardware | A GPU (a Kaggle T4 is enough). A `SMOKE` test run needs only a CPU. |
| Cost | GPU time only: about 40 minutes of a T4. |

In [ ]:
SMOKE = False      # True: a quick test on a CPU with espeak-ng and a few built-in sentences, not a dataset to train from. False: the real run with OmniVoice on a GPU.
N_SAMPLES = 12     # SMOKE only: how many of the built-in sentences to speak (at most 12)
RUN_ID = "1"       # the number of this run, 1 to 16; parallel runs need different numbers, and with Tatoeba each number speaks its own sentences

## 1. Settings

Change nothing for an English dataset. For another language, type its code into `TTS_LANG`.

In [ ]:
TTS_LANG     = "en"     # the language: "en", "pt", "de", "es", "fr" ...
N_PROMPTS    = 2000     # how many sentences to speak
VOICE_PROMPT = ""       # describe a voice to invent, e.g. "male, middle-aged, low pitch"; empty: a young woman
REF_AUDIO    = ""       # or copy a voice: a recording of 3 to 10 seconds (a file or a link), or the folder of an earlier run
REF_TEXT     = ""       # the exact words spoken in the recording; not needed for the folder of an earlier run

### Advanced settings

Most people skip the next cell. A setting in it can also come from outside the notebook: if an environment variable has the same name, for example `TTS_LANG=de`, the notebook uses its value instead of the one written in the cell.

In [ ]:
# Helper code, hidden by default: reads a setting from an environment variable when one is set. Nothing here needs editing.
import json, os, re, time, zlib
from collections import Counter
from pathlib import Path


def setting(name, default):
    # the environment variable `name` when it is set, else `default`; the value keeps the type of `default`
    raw = os.environ.get(name)
    if raw is None:
        return default
    if isinstance(default, bool):
        return raw.strip().lower() in ("1", "true", "yes")
    return type(default)(raw)

In [ ]:
# What to speak (the first lines pick up the Settings cell)
TTS_LANG      = setting("TTS_LANG", TTS_LANG)        # language code (en, pt, de, es-ES ...)
N_PROMPTS     = setting("N_PROMPTS", N_PROMPTS)      # how many sentences to draw from Tatoeba
MIN_WORDS     = setting("MIN_WORDS", 4)          # a drawn sentence has at least this many words
MAX_WORDS     = setting("MAX_WORDS", 20)         # and at most this many
PROMPT_SHARDS = setting("PROMPT_SHARDS", 16)     # the sentences are split into this many groups; a run speaks one group
PROMPT_SHARD  = setting("PROMPT_SHARD", "")      # this run's group, 0 to PROMPT_SHARDS - 1; empty: RUN_ID - 1. Your own sentences are split only when it is set
PROMPTS_FILE  = setting("PROMPTS_FILE", "")      # your own sentences: one per line, or piper-recording-studio "id<TAB>text"
PROMPTS_DIR   = setting("PROMPTS_DIR", "")       # or a folder of such .txt files
PROMPTS_CSV   = setting("PROMPTS_CSV", "")       # or a CSV/TSV file with a text column (LJSpeech "id|text" is read too)
CSV_TEXT_COL  = setting("CSV_TEXT_COL", "text")
CSV_SEP       = setting("CSV_SEP", "|")
PROMPTS_SOURCE_NAME = setting("PROMPTS_SOURCE_NAME", "")   # a name for the sentence source, used in clip ids; empty: its own name

# The voice: set REF_AUDIO and REF_TEXT, or VOICE_PROMPT, or nothing
REF_AUDIO     = setting("REF_AUDIO", REF_AUDIO).strip()     # a recording to copy the voice from: a path or a URL
REF_TEXT      = setting("REF_TEXT", REF_TEXT).strip()      # the exact words spoken in REF_AUDIO
VOICE_PROMPT  = setting("VOICE_PROMPT", VOICE_PROMPT).strip()  # a description of a voice to invent
DONOR_MODEL   = setting("DONOR_MODEL", "").strip()   # a phoonnx catalog voice id or an OVOS TTS plugin name; replaces OmniVoice
SEED          = setting("SEED", 42)                  # the random numbers of the sentence draw and of the voice
BATCH_SIZE    = setting("BATCH_SIZE", 16)             # sentences OmniVoice speaks per call
NUM_STEP      = setting("NUM_STEP", 16)               # OmniVoice decoding steps per clip; fewer is faster and less clean

# The run and its folders
RUN_ID        = setting("RUN_ID", str(RUN_ID)).strip()
OUTPUT_DIR    = setting("OUTPUT_DIR", f"./tts_dataset_run_{RUN_ID}")     # the dataset
# The donor's raw audio; not part of the result. On Kaggle it goes outside the Output tab.
WORK_DIR      = setting("WORK_DIR", f"/tmp/tts_dataset_work_{RUN_ID}" if os.path.isdir("/kaggle") else f"./tts_dataset_work_{RUN_ID}")
MANIFEST_NAME = setting("MANIFEST_NAME", "manifest.jsonl")

# Post-processing
TARGET_SR     = setting("TARGET_SR", 24000)      # sample rate of the clips (OmniVoice speaks at 24 kHz)
TRIM_SILENCE  = setting("TRIM_SILENCE", True)    # cut the silence at both ends
TRIM_TOP_DB   = setting("TRIM_TOP_DB", 30)       # how far below the peak counts as silence, in dB
TARGET_DBFS   = setting("TARGET_DBFS", -20.0)    # loudness every clip is brought to
MIN_SECONDS   = setting("MIN_SECONDS", 0.5)      # a shorter clip is dropped
MAX_SECONDS   = setting("MAX_SECONDS", 20.0)     # a longer clip is dropped
SR_ENGINE     = setting("SR_ENGINE", "").strip().lower()   # optional audiosronnx super-resolution engine; empty: off
SR_CACHE_DIR  = setting("SR_CACHE_DIR", "")      # where audiosronnx keeps downloaded weights; empty: its own default
SR_REVISION   = setting("SR_REVISION", "")       # a Hugging Face revision of the SR_ENGINE weights
ONNX_PROVIDERS = setting("ONNX_PROVIDERS", "auto")   # onnxruntime providers for SR_ENGINE: a comma list, or auto

In [ ]:
# Helper code, hidden by default: checks the settings and works out which sentences this run speaks. Nothing here needs editing.
# OmniVoice files four languages of the Tatoeba table under other codes, and does not speak four others of it.
OMNIVOICE_CODES = {"ar": "arb", "ku": "kmr", "ne": "npi", "tl": "fil"}
NOT_IN_OMNIVOICE = {"fo", "gd", "io", "la"}


def omnivoice_language(tag, lang_ids, name_to_id):
    # the OmniVoice code of a language tag or language name, or None
    primary = tag.replace("_", "-").split("-")[0].lower()
    primary = OMNIVOICE_CODES.get(primary, primary)
    if primary in lang_ids:
        return primary
    return name_to_id.get(tag.strip().lower())


def voice_in_folder(folder):
    # the voice_ref.wav of an earlier run somewhere inside the folder, and the words in the voice_ref.txt beside it ("" without one)
    found = sorted(Path(folder).rglob("voice_ref.wav"))
    if len({p.read_bytes() for p in found}) != 1:
        raise RuntimeError(f"REF_AUDIO={str(folder)!r} is a folder with {'no' if not found else 'more than one'} voice in it "
                           f"({', '.join(map(str, found)) or 'no voice_ref.wav'}). Point it at one voice_ref.wav file.")
    words = found[0].with_suffix(".txt")
    return str(found[0]), words.read_text(encoding="utf-8").strip() if words.is_file() else ""


def check_settings():
    # stops on settings that contradict each other; fixes the sentence group of the run
    global _STEPS, DATASET_FORMAT, DONOR_MODE, LANG_PRIMARY, REF_AUDIO, REF_TEXT, SHARD, SHARD_FROM, tick
    assert re.fullmatch(r"[A-Za-z0-9_-]+", RUN_ID), "RUN_ID may hold only letters, digits, '-' and '_'."
    DATASET_FORMAT = 1                               # the value of "tts_dataset" in dataset.json; train_vits checks it
    LANG_PRIMARY = TTS_LANG.replace("_", "-").split("-")[0].lower()
    DONOR_MODE = "model" if DONOR_MODEL else ("ci" if SMOKE else "omnivoice")
    if DONOR_MODE == "omnivoice" and LANG_PRIMARY in NOT_IN_OMNIVOICE:
        raise RuntimeError(f"OmniVoice does not support TTS_LANG={TTS_LANG!r}. Set DONOR_MODEL in the advanced settings to a "
                           f"phoonnx voice or an OVOS TTS plugin that speaks it.")

    if DONOR_MODE != "omnivoice" and (REF_AUDIO or REF_TEXT or VOICE_PROMPT):
        raise RuntimeError("REF_AUDIO, REF_TEXT and VOICE_PROMPT choose the OmniVoice voice; they do nothing "
                           "with DONOR_MODEL or SMOKE=True. Unset them or unset DONOR_MODEL.")
    if REF_AUDIO and VOICE_PROMPT:
        raise RuntimeError("Set REF_AUDIO (copy a voice) or VOICE_PROMPT (invent a voice), not both.")
    if REF_AUDIO and os.path.isdir(REF_AUDIO):
        REF_AUDIO, _words = voice_in_folder(REF_AUDIO)
        REF_TEXT = REF_TEXT or _words
        if not REF_TEXT:
            _txt = Path(REF_AUDIO).with_suffix(".txt")
            raise RuntimeError(f"{_txt} {'is empty' if _txt.is_file() else 'does not exist'}, so the words of {REF_AUDIO} "
                               f"are unknown. Set REF_TEXT to the exact words spoken in that recording.")
    elif REF_AUDIO and not re.match(r"^https?://", REF_AUDIO) and not os.path.isfile(REF_AUDIO):
        raise FileNotFoundError(f"REF_AUDIO={REF_AUDIO!r} does not exist. On Kaggle, files you add are under /kaggle/input.")
    if REF_AUDIO and not REF_TEXT:
        raise RuntimeError("REF_AUDIO needs REF_TEXT, the exact words spoken in the recording. The notebook does not "
                           "transcribe audio: without the words OmniVoice would load a Whisper model.")
    if REF_TEXT and not REF_AUDIO:
        raise RuntimeError("REF_TEXT is the words of REF_AUDIO; set REF_AUDIO too.")
    assert PROMPT_SHARDS >= 1, "PROMPT_SHARDS must be at least 1."
    assert N_PROMPTS >= 1, "N_PROMPTS must be at least 1."
    _shard = str(PROMPT_SHARD).strip()
    if _shard:
        if not re.fullmatch(r"[0-9]+", _shard) or int(_shard) >= PROMPT_SHARDS:
            raise ValueError(f"PROMPT_SHARD={PROMPT_SHARD!r} is not a group: give a whole number from 0 to {PROMPT_SHARDS - 1}.")
        SHARD, SHARD_FROM = int(_shard), "PROMPT_SHARD"
    elif re.fullmatch(r"[1-9][0-9]*", RUN_ID) and int(RUN_ID) <= PROMPT_SHARDS:
        SHARD, SHARD_FROM = int(RUN_ID) - 1, "RUN_ID"
    else:
        raise ValueError(f"RUN_ID={RUN_ID!r} must be a whole number from 1 to {PROMPT_SHARDS} without a leading zero, "
                         f"unless PROMPT_SHARD is set.")

    _out = Path(OUTPUT_DIR).resolve()
    for _ds in _out.glob("*/**/dataset.json"):
        try:
            _same_run = str(json.loads(_ds.read_text(encoding="utf-8")).get("run_id")) == RUN_ID
        except (ValueError, OSError, AttributeError):
            _same_run = False
        if _same_run and (_ds.parent / "voice_ref.wav").is_file():
            raise RuntimeError(f"OUTPUT_DIR={OUTPUT_DIR!r} already holds run {RUN_ID} with its voice in {_ds.parent}; "
                               f"give this run another RUN_ID or another OUTPUT_DIR.")

    _STEPS, _last = {}, [time.time()]
    def tick(name):
        # records the seconds spent since the previous tick, for the run-time table at the end
        now = time.time()
        _STEPS[name] = now - _last[0]
        _last[0] = now

    print(f"Run {RUN_ID}: language {TTS_LANG}, folder {OUTPUT_DIR}")
    if DONOR_MODE == "omnivoice":
        print("Voice: OmniVoice, " + (f"copied from {REF_AUDIO}" if REF_AUDIO else f"invented from {VOICE_PROMPT!r}" if VOICE_PROMPT
                                     else "invented from the default description"))
    elif DONOR_MODE == "model":
        print(f"Voice: DONOR_MODEL={DONOR_MODEL}")
    else:
        print("Voice: espeak-ng (the SMOKE test only; not a voice to build a dataset from)")

check_settings()

## 2. Install and check the machine

Installs the speech model and checks that the GPU works. A network error here means Internet is off.

In [ ]:
import json, shlex, shutil, subprocess, sys

_pkgs = ['"librosa>=0.10.2"', '"soundfile>=0.14.0"', '"numpy>=2.0"', '"tqdm>=4.70.1"']
if DONOR_MODE == "omnivoice":
    _pkgs.append('"omnivoice>=0.2.1"')
elif DONOR_MODE == "model":
    _pkgs += ['"ovos-plugin-manager>=2.12.4a1"', '"phoonnx"', '"nest-asyncio>=1.6.0"']
%pip install --quiet --pre {" ".join(_pkgs)}

import numpy as np
import soundfile as sf
from tqdm import tqdm

if DONOR_MODE == "omnivoice":
    try:
        import torch
    except ImportError as exc:
        raise ImportError("torch is not installed. Kaggle and Colab have it; elsewhere run: pip install torch") from exc
    try:
        (torch.ones(8, device="cuda") * 2).sum().item()
    except Exception as exc:
        raise RuntimeError(
            f"OmniVoice needs a GPU and no GPU can run a calculation here ({exc}). On Kaggle: Settings panel, "
            f"Accelerator, GPU T4, then Run All again. For a CPU-only run set DONOR_MODEL."
        ) from exc
    print(f"torch {torch.__version__} | {torch.cuda.get_device_name(0)}")
elif DONOR_MODE == "ci":
    assert shutil.which("espeak-ng"), "SMOKE speaks with espeak-ng, which is not installed (Linux: apt-get install espeak-ng)."
print(f"Python {sys.version.split()[0]} | donor: {DONOR_MODE}")

out_dir = Path(OUTPUT_DIR).resolve()
raw_dir = Path(WORK_DIR) / "donor_raw"
wavs_dir = out_dir / "wavs"
for _d in (raw_dir, wavs_dir):          # a re-run replaces its own earlier clips
    shutil.rmtree(_d, ignore_errors=True)
    _d.mkdir(parents=True)
free_gb = shutil.disk_usage(out_dir).free / 1e9
assert free_gb > (0.2 if SMOKE else 3), f"Only {free_gb:.1f} GB of disk is free; this run needs about {0.2 if SMOKE else 3} GB."
print(f"Disk free: {free_gb:.1f} GB")
tick("install and setup")

## 3. Make the dataset

Five steps: choose the sentences, speak them in one voice, tidy the clips, write the folder, check it. The speaking step is the long one.

In [ ]:
# Helper code, hidden by default: reads Tatoeba (or your own file) and keeps short, clean sentences. Nothing here needs editing.
DEMO_PROMPTS = {
    "en": [
        "The old lighthouse keeper climbed the stairs every evening to light the lamp.",
        "Please remind me to water the plants before I leave for the weekend.",
        "A gentle breeze moved through the orchard as the sun slipped behind the hills.",
        "The train to the coast leaves at a quarter past eight from platform four.",
        "She opened the window and listened to the rain tapping on the roof.",
        "Could you turn the volume down a little, the baby is finally asleep?",
        "Fresh bread, warm butter and a cup of tea made the morning feel slow and kind.",
        "The museum is open every day except Monday, and children enter for free.",
        "He carefully folded the map and tucked it into the pocket of his coat.",
        "Open source software belongs to everyone who takes the time to improve it.",
        "After a long walk along the river, we stopped at a small cafe for lunch.",
        "The weather tomorrow will be cloudy in the morning, with sunshine after noon.",
    ],
    "es": [
        "El viejo farero subía las escaleras cada tarde para encender la lámpara.",
        "Por favor, recuérdame regar las plantas antes de irme el fin de semana.",
        "Una brisa suave recorría el huerto mientras el sol se escondía tras las colinas.",
        "El tren hacia la costa sale a las ocho y cuarto desde el andén cuatro.",
        "Ella abrió la ventana y escuchó la lluvia golpeando el tejado.",
        "¿Podrías bajar un poco el volumen? El bebé por fin se ha dormido.",
        "Pan fresco, mantequilla tibia y una taza de té hacían la mañana lenta y amable.",
        "El museo abre todos los días menos los lunes, y los niños entran gratis.",
        "Dobló con cuidado el mapa y lo guardó en el bolsillo del abrigo.",
        "El software libre pertenece a todos los que dedican tiempo a mejorarlo.",
        "Después de un largo paseo junto al río, paramos en un café pequeño para comer.",
        "Mañana estará nublado por la mañana, con sol después del mediodía.",
    ],
    "pt": [
        "O velho faroleiro subia as escadas todas as tardes para acender a lâmpada.",
        "Por favor, lembra-me de regar as plantas antes de sair no fim de semana.",
        "Uma brisa suave passava pelo pomar enquanto o sol se escondia atrás das colinas.",
        "O comboio para a costa parte às oito e um quarto da linha quatro.",
        "Ela abriu a janela e ficou a ouvir a chuva a bater no telhado.",
        "Podes baixar um pouco o volume? O bebé finalmente adormeceu.",
        "Pão fresco, manteiga morna e uma chávena de chá deixavam a manhã lenta e simpática.",
        "O museu abre todos os dias, exceto à segunda-feira, e as crianças entram de graça.",
        "Dobrou com cuidado o mapa e guardou-o no bolso do casaco.",
        "O software livre pertence a todos os que dedicam tempo a melhorá-lo.",
        "Depois de uma longa caminhada junto ao rio, parámos num café pequeno para almoçar.",
        "Amanhã o tempo estará nublado de manhã, com sol depois do meio-dia.",
    ],
    "fr": [
        "Le vieux gardien du phare montait l'escalier chaque soir pour allumer la lampe.",
        "Rappelle-moi d'arroser les plantes avant de partir pour le week-end, s'il te plaît.",
        "Une brise légère traversait le verger pendant que le soleil disparaissait derrière les collines.",
        "Le train pour la côte part à huit heures quinze du quai numéro quatre.",
        "Elle ouvrit la fenêtre et écouta la pluie tambouriner sur le toit.",
        "Pourrais-tu baisser un peu le volume ? Le bébé vient enfin de s'endormir.",
        "Du pain frais, du beurre tiède et une tasse de thé rendaient le matin lent et doux.",
        "Le musée est ouvert tous les jours sauf le lundi, et l'entrée est gratuite pour les enfants.",
        "Il plia soigneusement la carte et la glissa dans la poche de son manteau.",
        "Les logiciels libres appartiennent à tous ceux qui prennent le temps de les améliorer.",
        "Après une longue promenade au bord de la rivière, nous nous sommes arrêtés dans un petit café pour déjeuner.",
        "Demain, le ciel sera nuageux le matin, avec du soleil l'après-midi.",
    ],
    "de": [
        "Der alte Leuchtturmwärter stieg jeden Abend die Treppe hinauf, um die Lampe anzuzünden.",
        "Bitte erinnere mich daran, die Pflanzen zu gießen, bevor ich am Wochenende wegfahre.",
        "Eine sanfte Brise zog durch den Obstgarten, während die Sonne hinter den Hügeln versank.",
        "Der Zug zur Küste fährt um Viertel nach acht von Gleis vier ab.",
        "Sie öffnete das Fenster und hörte dem Regen auf dem Dach zu.",
        "Könntest du die Lautstärke etwas senken? Das Baby ist endlich eingeschlafen.",
        "Frisches Brot, warme Butter und eine Tasse Tee machten den Morgen langsam und freundlich.",
        "Das Museum ist jeden Tag außer montags geöffnet, und Kinder haben freien Eintritt.",
        "Er faltete die Karte sorgfältig zusammen und steckte sie in die Manteltasche.",
        "Freie Software gehört allen, die sich die Zeit nehmen, sie zu verbessern.",
        "Nach einem langen Spaziergang am Fluss machten wir in einem kleinen Café Mittagspause.",
        "Morgen ist es vormittags bewölkt, am Nachmittag scheint die Sonne.",
    ],
    "it": [
        "Il vecchio guardiano del faro saliva le scale ogni sera per accendere la lampada.",
        "Per favore, ricordami di annaffiare le piante prima di partire per il fine settimana.",
        "Una brezza leggera attraversava il frutteto mentre il sole scompariva dietro le colline.",
        "Il treno per la costa parte alle otto e un quarto dal binario quattro.",
        "Lei aprì la finestra e ascoltò la pioggia che batteva sul tetto.",
        "Potresti abbassare un po' il volume? Il bambino si è finalmente addormentato.",
        "Pane fresco, burro caldo e una tazza di tè rendevano la mattina lenta e gentile.",
        "Il museo è aperto tutti i giorni tranne il lunedì, e i bambini entrano gratis.",
        "Piegò con cura la mappa e la infilò nella tasca del cappotto.",
        "Il software libero appartiene a tutti coloro che dedicano tempo a migliorarlo.",
        "Dopo una lunga passeggiata lungo il fiume, ci siamo fermati in un piccolo bar per pranzo.",
        "Domani il cielo sarà nuvoloso al mattino, con il sole nel pomeriggio.",
    ],
}

# Tatoeba files its sentences under ISO 639-3 codes; these are the two-letter codes it is known by.
TATOEBA_CODES = dict(item.split(":") for item in (
    "af:afr am:amh an:arg ar:ara az:aze ba:bak be:bel bg:bul bn:ben br:bre bs:bos ca:cat cs:ces cv:chv cy:cym da:dan de:deu "
    "el:ell en:eng eo:epo es:spa et:est eu:eus fa:pes fi:fin fo:fao fr:fra fy:fry ga:gle gd:gla gl:glg gu:guj ha:hau he:heb "
    "hi:hin hr:hrv hu:hun hy:hye ia:ina id:ind ig:ibo io:ido is:isl it:ita ja:jpn jv:jav ka:kat kk:kaz km:khm kn:kan ko:kor "
    "ku:kmr ky:kir la:lat lb:ltz lo:lao lt:lit lv:lvs mk:mkd ml:mal mn:mon mr:mar ms:zsm mt:mlt my:mya nb:nob ne:npi nl:nld "
    "nn:nno no:nob oc:oci os:oss pa:pan pl:pol ps:pus pt:por ro:ron ru:rus si:sin sk:slk sl:slv so:som sq:sqi sr:srp sv:swe "
    "sw:swh ta:tam te:tel th:tha tk:tuk tl:tgl tr:tur tt:tat ug:uig uk:ukr ur:urd uz:uzb vi:vie xh:xho yi:yid yo:yor zh:cmn "
    "zu:zul").split())
TATOEBA_URL = "https://downloads.tatoeba.org/exports/per_language/{code}/{code}_sentences.tsv.bz2"
UNSPACED = {"cmn", "jpn", "tha", "khm", "lao", "mya", "yue", "wuu"}      # written without spaces: two characters count as a word
_SENTENCE_RE = re.compile(r"^[¿¡]?[^\W\d_][^\d\"“”„«»‹›()\[\]{}<>/\\|=+*#@&%$€£^~_`]*[.!?。！？؟।։።]$")
_BRACKET_RE = re.compile(r"\[[^\]]*\]|<[^>]*>")


def _clean_line(text):
    return re.sub(r"\s+", " ", text.strip())


def _read_lines_file(path):
    # one sentence per line, or piper-recording-studio "id<TAB>text"
    lines = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if "\t" in line:
                line = line.split("\t", 1)[1]
            lines.append(_clean_line(line))
    return lines


def _read_lines_dir(folder):
    lines = []
    for txt in sorted(Path(folder).rglob("*.txt")):
        lines.extend(_read_lines_file(txt))
    return lines


def _read_lines_csv(path, text_col, sep):
    import csv
    lines = []
    with open(path, "r", encoding="utf-8") as f:
        sample = f.read(4096)
        f.seek(0)
        has_header = bool(sample.strip()) and csv.Sniffer().has_header(sample)
        reader = csv.reader(f, delimiter=sep)
        if has_header:
            header = [h.strip().lower() for h in next(reader)]
            col_idx = header.index(text_col.lower()) if text_col.lower() in header else min(1, len(header) - 1)
        else:
            col_idx = 1 if sep == "|" else 0
        for row in reader:
            if len(row) > col_idx:
                lines.append(_clean_line(row[col_idx]))
    return lines


def _text_refusal_reason(text):
    if not text or not text.strip():
        return "empty"
    if _BRACKET_RE.search(text):
        return "bracketed_annotation"
    if not any(ch.isalpha() for ch in text):
        return "no_letter"
    if "|" in text:
        return "pipe_character"
    return None


def _in_shard(text, shard=None):
    # the group of a sentence depends on its text alone, so it is the same in every run
    return zlib.crc32(text.encode("utf-8")) % PROMPT_SHARDS == (SHARD if shard is None else shard)


def _demo_prompt_lines(lang, n):
    if lang not in DEMO_PROMPTS:
        raise RuntimeError(f"The built-in sentences cover only {', '.join(sorted(DEMO_PROMPTS))}, not {lang!r}. "
                           f"Set PROMPTS_FILE, PROMPTS_DIR or PROMPTS_CSV.")
    return DEMO_PROMPTS[lang][:max(1, min(int(n), len(DEMO_PROMPTS[lang])))]


def _tatoeba_sentences(code):
    # every sentence of one language, read from the compressed file as it arrives; nothing is written to disk
    import bz2, http.client, urllib.error, urllib.request
    url, last = TATOEBA_URL.format(code=code), None
    for attempt in range(3):
        if attempt:
            time.sleep(5 * attempt)
        try:
            unpack, parts, n_bytes = bz2.BZ2Decompressor(), [], 0
            with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "ovos-ngi0-notebooks"}), timeout=120) as resp:
                while chunk := resp.read(1 << 20):
                    n_bytes += len(chunk)
                    parts.append(unpack.decompress(chunk))
            if not unpack.eof:
                raise EOFError(f"the download stopped after {n_bytes} bytes, before the end of the file")
            rows = b"".join(parts).decode("utf-8").splitlines()
            return [_clean_line(r.split("\t", 2)[2]) for r in rows if r.count("\t") >= 2], n_bytes
        except urllib.error.HTTPError as exc:
            if exc.code == 404:
                raise RuntimeError(f"Tatoeba has no sentences filed under {code!r} (from TTS_LANG={TTS_LANG!r}). Give the "
                                   f"three-letter code Tatoeba uses as TTS_LANG, or set PROMPTS_FILE to your own sentences.") from exc
            last = exc
        except (OSError, EOFError, http.client.IncompleteRead) as exc:
            last = exc
    raise RuntimeError(f"Could not read {url} ({last}). Check that Internet is on (Kaggle: Settings panel, Internet), "
                       f"or set PROMPTS_FILE to your own sentences.")


def _n_words(text, code):
    return len(text) // 2 if code in UNSPACED else len(text.split())


def _draw(sentences, code, n, seed, shard=None):
    # n sentences that pass the rules above, of this run's group, in a random order fixed by the seed
    import random
    seen, pool = set(), []
    for s in sentences:
        key = s.casefold()
        if (key in seen or not MIN_WORDS <= _n_words(s, code) <= MAX_WORDS or not _SENTENCE_RE.match(s) or not _in_shard(s, shard)
                or any(len(w) > 1 and w.isupper() for w in s.split())):      # an acronym is spelled, not read
            continue
        seen.add(key)
        pool.append(s)
    pool.sort()
    random.Random(seed).shuffle(pool)
    cap, names, chosen, passed_over = max(3, n // 50), Counter(), [], 0
    for s in pool:
        if len(chosen) == n:
            break
        words = {w.strip(".,!?;:'’") for w in s.split()[1:] if w[:1].isupper()}
        if any(names[w] >= cap for w in words):
            passed_over += 1
            continue
        names.update(words)
        chosen.append(s)
    return chosen, len(pool), passed_over


def choose_sentences():
    # fills PROMPT_ROWS with the sentences of this run
    global PROMPT_LICENCE, PROMPT_ROWS, prompts_source_kind, prompts_source_name, prompts_source_path, raw_lines, \
        reason_counts, shard_applies
    n_set = sum(1 for p in (PROMPTS_CSV, PROMPTS_DIR, PROMPTS_FILE) if p)
    if n_set > 1:
        raise RuntimeError("Set at most one of PROMPTS_FILE, PROMPTS_DIR and PROMPTS_CSV.")

    PROMPT_LICENCE = None
    shard_applies = bool(str(PROMPT_SHARD).strip())           # your own sentences are split only on request
    if PROMPTS_CSV:
        if not os.path.exists(PROMPTS_CSV):
            raise FileNotFoundError(f"PROMPTS_CSV={PROMPTS_CSV!r} does not exist")
        raw_lines = _read_lines_csv(PROMPTS_CSV, CSV_TEXT_COL, CSV_SEP)
        prompts_source_kind, prompts_source_path = "csv", PROMPTS_CSV
        prompts_source_name = PROMPTS_SOURCE_NAME or Path(PROMPTS_CSV).stem
    elif PROMPTS_DIR:
        if not os.path.isdir(PROMPTS_DIR):
            raise FileNotFoundError(f"PROMPTS_DIR={PROMPTS_DIR!r} is not a directory")
        raw_lines = _read_lines_dir(PROMPTS_DIR)
        prompts_source_kind, prompts_source_path = "dir", PROMPTS_DIR
        prompts_source_name = PROMPTS_SOURCE_NAME or Path(PROMPTS_DIR).name
    elif PROMPTS_FILE:
        if not os.path.exists(PROMPTS_FILE):
            raise FileNotFoundError(f"PROMPTS_FILE={PROMPTS_FILE!r} does not exist")
        raw_lines = _read_lines_file(PROMPTS_FILE)
        prompts_source_kind, prompts_source_path = "file", PROMPTS_FILE
        prompts_source_name = PROMPTS_SOURCE_NAME or Path(PROMPTS_FILE).stem
    elif SMOKE:
        raw_lines = _demo_prompt_lines(LANG_PRIMARY, N_SAMPLES)
        prompts_source_kind, prompts_source_path = "demo", None
        prompts_source_name = PROMPTS_SOURCE_NAME or f"demo-{LANG_PRIMARY}"
        print(f"SMOKE: {len(raw_lines)} built-in sentences ({LANG_PRIMARY}).")
        # These extra SMOKE test sentences exist to show refusal working, not merely to run:
        # a rule that stopped refusing must fail this cell, not print a quiet count.
        raw_lines = raw_lines + [
            "[inaudible]",           # refused: bracketed annotation
            "12345",                 # refused: no letter
            '"Hello there," she said to the old keeper.',   # kept: its quotes must come back intact from metadata.csv
        ]
    else:
        tatoeba_code = TATOEBA_CODES.get(LANG_PRIMARY, LANG_PRIMARY)
        if len(tatoeba_code) != 3:
            raise RuntimeError(f"No Tatoeba code is known for TTS_LANG={TTS_LANG!r}. Give the three-letter code Tatoeba uses "
                               f"(https://tatoeba.org/en/stats/sentences_by_language), or set PROMPTS_FILE to your own sentences.")
        t0 = time.time()
        all_sentences, n_bytes = _tatoeba_sentences(tatoeba_code)
        raw_lines, n_pool, n_passed_over = _draw(all_sentences, tatoeba_code, N_PROMPTS, SEED)
        print(f"Tatoeba {tatoeba_code}: {len(all_sentences)} sentences read ({n_bytes / 1e6:.1f} MB in {time.time() - t0:.0f} s); "
              f"{n_pool} pass the rules in group {SHARD} of {PROMPT_SHARDS} (from {SHARD_FROM}); {len(raw_lines)} drawn, "
              f"{n_passed_over} passed over for a repeated name")
        if len(raw_lines) < N_PROMPTS:
            raise RuntimeError(f"Tatoeba has only {len(raw_lines)} usable sentences for {tatoeba_code!r} in this run's group; "
                               f"N_PROMPTS asks for {N_PROMPTS}. Lower N_PROMPTS, "
                               + ("set PROMPT_SHARDS to 1, " if PROMPT_SHARDS > 1 else "")
                               + "or use your own sentences.")
        del all_sentences
        prompts_source_kind, prompts_source_path = "tatoeba", TATOEBA_URL.format(code=tatoeba_code)
        prompts_source_name = PROMPTS_SOURCE_NAME or f"tatoeba-{tatoeba_code}"
        PROMPT_LICENCE = "CC BY 2.0 FR (https://tatoeba.org)"
        shard_applies = True

    reason_counts = Counter()
    PROMPT_ROWS = []
    for text in raw_lines:
        reason = _text_refusal_reason(text)
        if not reason and shard_applies and prompts_source_kind != "tatoeba" and not _in_shard(text):
            reason = "other_shard"
        if reason:
            reason_counts[reason] += 1
            continue
        PROMPT_ROWS.append({"id": f"{prompts_source_name}-r{RUN_ID}-{len(PROMPT_ROWS):06d}", "text": text})
        reason_counts["passed"] += 1

    print(f"Sentences read: {len(raw_lines)}")
    for reason in ("empty", "bracketed_annotation", "no_letter", "pipe_character", "other_shard", "passed"):
        if reason_counts.get(reason) or reason == "passed":
            print(f"  {reason:22s} {reason_counts.get(reason, 0)}")
    if not PROMPT_ROWS:
        raise RuntimeError("Every sentence was refused, or none were loaded; nothing to speak.")
    print("For example:")
    for _row in PROMPT_ROWS[:5]:
        print(f"  {_row['text']}")

    if SMOKE and prompts_source_kind == "demo":
        assert reason_counts.get("bracketed_annotation", 0) >= 1, "the SMOKE test's bracketed-annotation sentence was not refused"
        assert reason_counts.get("no_letter", 0) >= 1, "the SMOKE test's letterless sentence was not refused"
        assert _text_refusal_reason("a|b") == "pipe_character", "a prompt with a pipe character was not refused"
        # the sentence rules and the groups, on fixed rows: every group is disjoint from the others and together they hold every kept row
        _fix = [f"Sentence number {w} is here." for w in "one two three four five six seven eight nine ten".split()] + ["Too short.", "It has 3 digits in it."]
        _groups = [set(_draw(_fix, "eng", 100, SEED, shard=k)[0]) for k in range(PROMPT_SHARDS)]
        assert sum(len(g) for g in _groups) == len(set().union(*_groups)) == 10, "the sentence groups overlap or lose a sentence"
        # the name cap: a name after the first word is in at most max(3, n // 50) of the drawn sentences
        _adj = "red blue green old new small big quiet busy empty dark bright".split()
        _named = [s for s in (f"The {a} dog saw Tom near the {b} gate." for a in _adj for b in _adj) if _in_shard(s, 0)]
        assert len(_named) > 3 and len(_draw(_named, "eng", 100, SEED, shard=0)[0]) == 3, "the name cap let one name into more than 3 of 100 sentences"
        assert _SENTENCE_RE.match("¿Dónde está la estación de tren?"), "a Spanish question that opens with ¿ was refused"
    tick("choose the sentences")

In [ ]:
# Helper code, hidden by default: loads OmniVoice, fixes one voice in voice_ref.wav and speaks every sentence with it. Nothing here needs editing.
import hashlib

# What the donor stage fills in for the manifest and the sidecar.
DONOR_PLUGIN_ID = DONOR_VOICE_ID = DONOR_VERSION = DONOR_NOTE = None
DONOR_DETAILS = {}
GENERATION = {}
SYNTH_ROWS = []
n_synth_failed = 0
VOICE_REF = out_dir / "voice_ref.wav"
VOICE_REF_TEXT = None

# The language is checked here, before the sentences and the model are downloaded.
OMNI_LANG = None
if DONOR_MODE == "omnivoice":
    from omnivoice.utils.lang_map import LANG_IDS, LANG_NAME_TO_ID
    OMNI_LANG = omnivoice_language(TTS_LANG, LANG_IDS, LANG_NAME_TO_ID)
    if OMNI_LANG is None:
        raise RuntimeError(
            f"OmniVoice does not support TTS_LANG={TTS_LANG!r}. It speaks {len(LANG_IDS)} languages "
            f"(omnivoice.utils.lang_map.LANG_NAME_TO_ID lists their names and codes; a variety is named by its "
            f"own code or name, for example arb or 'standard arabic'). If the language is not on the list, "
            f"set DONOR_MODEL to a model from the phoonnx catalog or to the name of an OVOS TTS plugin."
        )

# The voices of the published dataset identities, which are conversion targets and never donors.
_REFUSED_VOICE_RE = re.compile(r"^openvoiceos/.*[_-](miro|dii)(?:[_-]|$)")


def _slug(s):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", s)


def _usable(audio):
    # False for an empty, non-finite or silent waveform
    audio = np.asarray(audio)
    return bool(audio.size) and bool(np.isfinite(audio).all()) and float(np.abs(audio).max()) > 1e-4


def _gpu_memory():
    out = {
        "peak_allocated_gib": round(torch.cuda.max_memory_allocated() / 2**30, 2),
        "peak_reserved_gib": round(torch.cuda.max_memory_reserved() / 2**30, 2),
        "total_gib": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
    }
    try:
        used = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                              capture_output=True, text=True, check=True).stdout.split()[0]
        out["nvidia_smi_used_gib"] = round(int(used) / 1024, 2)
    except (OSError, subprocess.CalledProcessError, IndexError, ValueError):
        pass
    return out


def _write_raw(row, audio, sr):
    path = raw_dir / f"{_slug(row['id'])}.wav"
    sf.write(str(path), np.asarray(audio, dtype=np.float32), sr)
    return path


def speak_sentences():
    # fills SYNTH_ROWS with one raw clip per sentence
    global DONOR_DETAILS, DONOR_NOTE, DONOR_PLUGIN_ID, DONOR_VERSION, DONOR_VOICE_ID, GENERATION, n_synth_failed, \
        voice_mode, VOICE_REF_TEXT
    if DONOR_MODE == "omnivoice":
        import random
        from importlib.metadata import version
        from omnivoice import OmniVoice
        from omnivoice.models.omnivoice import OmniVoiceGenerationConfig

        def _seed_all(value):
            random.seed(value)
            np.random.seed(value)
            torch.manual_seed(value)
            torch.cuda.manual_seed_all(value)

        torch.cuda.reset_peak_memory_stats()
        gpu_name = torch.cuda.get_device_name(0)
        t0 = time.time()
        tts = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map="cuda:0", dtype=torch.float16)
        load_seconds = time.time() - t0
        gpu_after_load = _gpu_memory()
        print(f"OmniVoice {version('omnivoice')} loaded on {gpu_name} in {load_seconds:.0f} s "
              f"({gpu_after_load['peak_allocated_gib']} GiB allocated)")
        gen_config = OmniVoiceGenerationConfig(num_step=NUM_STEP)

        if REF_AUDIO:
            assert REF_TEXT, "REF_TEXT is required with REF_AUDIO"   # a missing transcript would make OmniVoice load its Whisper model
            if re.match(r"^https?://", REF_AUDIO):
                import urllib.request
                ref_path = Path(WORK_DIR) / ("reference" + (Path(REF_AUDIO.split("?")[0]).suffix or ".wav"))
                urllib.request.urlretrieve(REF_AUDIO, ref_path)
            else:
                ref_path = Path(REF_AUDIO)
                if not ref_path.is_file():
                    raise FileNotFoundError(f"REF_AUDIO={REF_AUDIO!r} does not exist")
            if ref_path.suffix.lower() == ".wav":
                if ref_path.resolve() != VOICE_REF:
                    shutil.copyfile(ref_path, VOICE_REF)         # byte for byte, so the voice id is the same in every run
            else:
                import librosa
                _wav, _sr = librosa.load(str(ref_path), sr=None, mono=True)
                sf.write(str(VOICE_REF), _wav, _sr, subtype="PCM_16")
            VOICE_REF_TEXT = REF_TEXT
            voice_mode = "clone"
            voice_detail = {"ref_audio": Path(REF_AUDIO.split("?")[0]).name}
        else:
            design = VOICE_PROMPT or "female, young adult, moderate pitch" + (", american accent" if OMNI_LANG == "en" else "")
            # Two sentences, long enough for a clean 5 to 12 s reference: the built-in ones where the language has them.
            anchor_text = ""
            for _text in DEMO_PROMPTS.get(LANG_PRIMARY, []) + [r["text"] for r in PROMPT_ROWS]:
                anchor_text = (anchor_text + " " + _text).strip()
                if len(anchor_text) >= 100:
                    break
            anchor_text = anchor_text[:300]
            _seed_all(SEED)
            anchor = tts.generate(text=anchor_text, language=OMNI_LANG, instruct=design, generation_config=gen_config)[0]
            if not _usable(anchor):
                raise RuntimeError(f"OmniVoice produced no audio for the voice description {design!r}.")
            sf.write(str(VOICE_REF), np.asarray(anchor, dtype=np.float32), tts.sampling_rate, subtype="PCM_16")
            VOICE_REF_TEXT = anchor_text
            voice_mode = "design"
            voice_detail = {"voice_prompt": design, "anchor_seconds": round(len(anchor) / tts.sampling_rate, 2)}
        (out_dir / "voice_ref.txt").write_text(VOICE_REF_TEXT + "\n", encoding="utf-8")
        # Every run copies the voice from the file, so a run that is handed this file hears exactly what this run heard.
        clone_prompt = tts.create_voice_clone_prompt(ref_audio=str(VOICE_REF), ref_text=VOICE_REF_TEXT)
        print(f"Voice: {voice_mode}, {json.dumps(voice_detail, ensure_ascii=False)}")

        def _generate(texts):
            # A batch with one degenerate item is retried item by item; an item that still fails is None.
            try:
                return tts.generate(text=texts, language=OMNI_LANG, voice_clone_prompt=clone_prompt, generation_config=gen_config)
            except Exception:
                out = []
                for t in texts:
                    try:
                        out.append(tts.generate(text=[t], language=OMNI_LANG, voice_clone_prompt=clone_prompt,
                                                generation_config=gen_config)[0])
                    except Exception:
                        out.append(None)
                return out

        batch_size = max(1, int(BATCH_SIZE))
        _seed_all(SEED + zlib.crc32(RUN_ID.encode()) % 100000)
        by_length = sorted(PROMPT_ROWS, key=lambda r: len(r["text"]))       # a batch of similar lengths wastes little on padding
        torch.cuda.synchronize()
        gen_seconds = 0.0
        audio_seconds = 0.0
        for start in tqdm(range(0, len(by_length), batch_size), desc="OmniVoice batches", unit="batch", file=sys.stdout, mininterval=30):
            batch = by_length[start:start + batch_size]
            t1 = time.time()
            out = _generate([r["text"] for r in batch])
            torch.cuda.synchronize()
            gen_seconds += time.time() - t1
            for row, audio in zip(batch, out):
                if audio is None or not _usable(audio):
                    n_synth_failed += 1
                    tqdm.write(f"  Failed {row['id']}: no usable audio")
                    continue
                audio_seconds += len(audio) / tts.sampling_rate
                SYNTH_ROWS.append({**row, "raw_audio": str(_write_raw(row, audio, tts.sampling_rate))})
        SYNTH_ROWS.sort(key=lambda r: r["id"])

        DONOR_PLUGIN_ID = "omnivoice"
        DONOR_VERSION = version("omnivoice")
        DONOR_NOTE = f"OmniVoice (k2-fsa/OmniVoice, fp16, {gpu_name}), voice mode {voice_mode}, language {OMNI_LANG}"
        DONOR_DETAILS = {"model": "k2-fsa/OmniVoice", "dtype": "float16", "language": OMNI_LANG, "voice_mode": voice_mode,
                         "seed": SEED, "batch_size": batch_size, "num_step": NUM_STEP, **voice_detail}
        GENERATION = {
            "device": gpu_name,
            "clips": len(SYNTH_ROWS),
            "seconds": round(gen_seconds, 2),
            "clips_per_s": round(len(SYNTH_ROWS) / gen_seconds, 3) if gen_seconds else None,
            "audio_seconds": round(audio_seconds, 2),
            "audio_s_per_s": round(audio_seconds / gen_seconds, 2) if gen_seconds else None,
            "model_load_seconds": round(load_seconds, 1),
            "gpu_memory": _gpu_memory(),
        }
        del tts, clone_prompt
        torch.cuda.empty_cache()

    elif DONOR_MODE == "model":
        if _REFUSED_VOICE_RE.search(DONOR_MODEL.lower()):
            raise RuntimeError(f"DONOR_MODEL={DONOR_MODEL!r} is the voice of a published dataset identity, "
                               f"which is a conversion target and never a donor.")
        import nest_asyncio
        nest_asyncio.apply()
        from ovos_plugin_manager.tts import load_tts_plugin
        from phoonnx.model_manager import TTSModelManager

        catalog = TTSModelManager()
        catalog.load()
        catalog.merge_default_voices()
        if DONOR_MODEL in catalog.voices:
            DONOR_PLUGIN_ID, DONOR_VOICE_ID = "ovos-tts-plugin-phoonnx", DONOR_MODEL
            plugin_config = {"lang": TTS_LANG, "voice": DONOR_MODEL}
            DONOR_NOTE = f"phoonnx catalog voice {DONOR_MODEL!r}"
        else:
            if DONOR_MODEL in ("ovos-tts-plugin-phoonnx", "phoonnx"):
                raise RuntimeError("DONOR_MODEL names the phoonnx plugin without a voice. Name one voice id from "
                                   "the phoonnx catalog (phoonnx-voices list-voices --lang <tag>).")
            DONOR_PLUGIN_ID, DONOR_VOICE_ID = DONOR_MODEL, ""
            plugin_config = {"lang": TTS_LANG}
            DONOR_NOTE = f"OVOS TTS plugin {DONOR_MODEL!r} with its default voice"
        print(f"Donor: {DONOR_NOTE}")

        donor_cls = load_tts_plugin(DONOR_PLUGIN_ID)
        if donor_cls is None:
            print(f"{DONOR_PLUGIN_ID!r} is not an installed OVOS TTS plugin; installing it from PyPI.")
            get_ipython().run_line_magic("pip", f"install --quiet --pre {shlex.quote(DONOR_PLUGIN_ID)}")
            import importlib
            importlib.invalidate_caches()
            donor_cls = load_tts_plugin(DONOR_PLUGIN_ID)
        if donor_cls is None:
            raise RuntimeError(f"DONOR_MODEL={DONOR_MODEL!r} is neither a phoonnx catalog voice id nor an OVOS "
                               f"TTS plugin that pip can install. If the plugin was just installed, restart the "
                               f"kernel and run again.")
        donor_plugin = donor_cls(config=plugin_config)
        DONOR_VERSION = None
        import importlib.metadata
        for _candidate in (donor_cls.__module__.split(".")[0], DONOR_PLUGIN_ID, DONOR_PLUGIN_ID.replace("_", "-")):
            try:
                DONOR_VERSION = importlib.metadata.version(_candidate)
                break
            except importlib.metadata.PackageNotFoundError:
                continue
        # The plugin's own declared format, never assumed to be WAV.
        donor_audio_ext = (getattr(donor_plugin, "audio_ext", None) or "wav").lstrip(".")
        print(f"Loaded donor plugin {DONOR_PLUGIN_ID!r} (package version {DONOR_VERSION or 'unknown'}, "
              f"audio_ext={donor_audio_ext!r})")

        t0 = time.time()
        for row in tqdm(PROMPT_ROWS, desc="Donor synthesis", unit="utt", file=sys.stdout):
            raw_path = raw_dir / f"{_slug(row['id'])}.{donor_audio_ext}"
            try:
                donor_plugin.get_tts(row["text"], str(raw_path), lang=TTS_LANG, voice=DONOR_VOICE_ID or None)
                if not raw_path.is_file() or raw_path.stat().st_size == 0:
                    raise RuntimeError("the plugin wrote no audio")
                SYNTH_ROWS.append({**row, "raw_audio": str(raw_path)})
            except Exception as exc:
                n_synth_failed += 1
                tqdm.write(f"  Failed {row['id']}: {exc}")
        gen_seconds = time.time() - t0
        GENERATION = {"device": "plugin default", "clips": len(SYNTH_ROWS), "seconds": round(gen_seconds, 2),
                      "clips_per_s": round(len(SYNTH_ROWS) / gen_seconds, 3) if gen_seconds else None}

    else:
        # SMOKE test only. espeak-ng needs no network and no GPU; it is not a donor to build a dataset from.
        espeak = shutil.which("espeak-ng")
        DONOR_PLUGIN_ID, DONOR_VOICE_ID = "espeak-ng", LANG_PRIMARY
        _ver = re.search(r"\d+(?:\.\d+)+", subprocess.run([espeak, "--version"], capture_output=True, text=True).stdout)
        DONOR_VERSION = _ver.group(0) if _ver else None
        DONOR_NOTE = "espeak-ng, the SMOKE test only: a CPU synthesiser that checks the notebook runs, not a donor for a dataset"
        print(f"Donor: {DONOR_NOTE}")
        t0 = time.time()
        for row in tqdm(PROMPT_ROWS, desc="Donor synthesis", unit="utt", file=sys.stdout):
            raw_path = raw_dir / f"{_slug(row['id'])}.wav"
            try:
                subprocess.run([espeak, "-v", LANG_PRIMARY, "-w", str(raw_path), row["text"]],
                               check=True, capture_output=True)
                audio, _sr = sf.read(str(raw_path), dtype="float32")
                if not _usable(audio):
                    raise RuntimeError("espeak-ng wrote silence")
                SYNTH_ROWS.append({**row, "raw_audio": str(raw_path)})
            except Exception as exc:
                n_synth_failed += 1
                tqdm.write(f"  Failed {row['id']}: {exc}")
        gen_seconds = time.time() - t0
        GENERATION = {"device": "cpu", "clips": len(SYNTH_ROWS), "seconds": round(gen_seconds, 2),
                      "clips_per_s": round(len(SYNTH_ROWS) / gen_seconds, 3) if gen_seconds else None}

    print(f"Spoke {len(SYNTH_ROWS)} of {len(PROMPT_ROWS)} sentence(s); {n_synth_failed} failed")
    if GENERATION:
        print(f"Generation: {json.dumps(GENERATION)}")
    if not SYNTH_ROWS:
        raise RuntimeError("The donor produced no audio.")
    if SMOKE and len(SYNTH_ROWS) != len(PROMPT_ROWS):
        raise RuntimeError(f"The SMOKE test spoke {len(SYNTH_ROWS)} of {len(PROMPT_ROWS)} sentences; every one must succeed.")
    tick("speak the sentences")

In [ ]:
# Helper code, hidden by default: trims silence, evens out the loudness and drops clips that are too short or too long. Nothing here needs editing.
def clean_clips():
    # fills FINAL_ROWS with the finished clips in wavs/
    global FINAL_ROWS, n_bad_length, n_post_failed
    import librosa

    sr_model = None
    if SR_ENGINE:
        import inspect
        from audiosronnx import load_sr
        from audiosronnx.base import get_engine

        def _parse_env_value(raw):
            low = raw.strip().lower()
            if low in ("true", "false"):
                return low == "true"
            for caster in (int, float):
                try:
                    return caster(raw)
                except ValueError:
                    pass
            return raw

        def _onnx_providers():
            raw = ONNX_PROVIDERS.strip()
            if raw.lower() != "auto":
                return [p.strip() for p in raw.split(",") if p.strip()]
            import onnxruntime as ort
            priority = ["CUDAExecutionProvider", "ROCMExecutionProvider", "MIGraphXExecutionProvider", "CPUExecutionProvider"]
            available = set(ort.get_available_providers())
            ordered = [p for p in priority if p in available]
            if "CPUExecutionProvider" not in ordered:
                ordered.append("CPUExecutionProvider")
            return ordered

        _entry = get_engine(SR_ENGINE)
        _sig = inspect.signature(_entry.adapter_class.__init__)
        _accepted = set(_sig.parameters) - {"self", "providers", "cache_dir", "revision"}
        sr_kwargs = {}
        for _name in _accepted:
            _raw = os.environ.get(f"SR_{_name.upper()}", "")
            if _raw:
                sr_kwargs[_name] = _parse_env_value(_raw)
        for _key in os.environ:
            if _key.startswith("SR_") and _key not in {"SR_ENGINE", "SR_CACHE_DIR", "SR_REVISION"}:
                _opt = _key[len("SR_"):].lower()
                if _opt not in _accepted:
                    raise RuntimeError(
                        f"{_key} is set but {SR_ENGINE!r} has no {_opt!r} option (accepts: {sorted(_accepted)})"
                    )

        sr_model = load_sr(
            SR_ENGINE,
            providers=_onnx_providers(),
            cache_dir=SR_CACHE_DIR or None,
            revision=SR_REVISION or None,
            **sr_kwargs,
        )
        print(f"Loaded audiosronnx SR engine {SR_ENGINE!r} with options {sr_kwargs}")

    FINAL_ROWS = []
    n_post_failed = 0
    n_bad_length = 0
    for row in tqdm(SYNTH_ROWS, desc="Post-processing", unit="utt", file=sys.stdout, mininterval=30):
        out_name = f"{_slug(row['id'])}.wav"
        out_path = wavs_dir / out_name
        try:
            wav, sr = librosa.load(row["raw_audio"], sr=None, mono=True)

            if sr_model is not None:
                wav, sr = sr_model.upscale(wav, sample_rate=sr)

            if sr != TARGET_SR:
                wav = librosa.resample(wav, orig_sr=sr, target_sr=TARGET_SR)
                sr = TARGET_SR

            if TRIM_SILENCE:
                wav, _ = librosa.effects.trim(wav, top_db=TRIM_TOP_DB)

            if not _usable(wav):
                raise RuntimeError("the clip is empty or silent after trimming")
            if not MIN_SECONDS <= len(wav) / sr <= MAX_SECONDS:
                n_bad_length += 1
                raise RuntimeError(f"the clip is {len(wav) / sr:.2f} s long, outside {MIN_SECONDS} to {MAX_SECONDS} s")

            rms = float(np.sqrt(np.mean(wav ** 2) + 1e-9))
            current_dbfs = 20 * np.log10(rms + 1e-9)
            gain = 10 ** ((TARGET_DBFS - current_dbfs) / 20.0)
            wav = np.clip(wav * gain, -1.0, 1.0).astype(np.float32)

            sf.write(str(out_path), wav, sr, subtype="PCM_16")
            FINAL_ROWS.append({
                **row,
                "audio": f"wavs/{out_name}",
                "duration_s": round(len(wav) / sr, 3),
                "sample_rate": int(sr),
            })
        except Exception as exc:
            n_post_failed += 1
            tqdm.write(f"  Dropped {row['id']}: {exc}")

    print(f"Post-processed {len(FINAL_ROWS)} of {len(SYNTH_ROWS)} clip(s); {n_post_failed} dropped, {n_bad_length} of them for their length")
    if not FINAL_ROWS:
        raise RuntimeError("Post-processing produced no audio.")
    if SMOKE and len(FINAL_ROWS) != len(SYNTH_ROWS):
        raise RuntimeError(f"The SMOKE test post-processed {len(FINAL_ROWS)} of {len(SYNTH_ROWS)} clips; every one must succeed.")
    tick("post-processing")

In [ ]:
# Helper code, hidden by default: writes metadata.csv, dataset.json and the manifest. Nothing here needs editing.
import csv

def write_dataset():
    # writes the text files of the folder
    global donor, DONOR_VOICE_ID, prompts_source, VOICE_REF_TEXT
    if DONOR_MODE != "omnivoice":
        # No recording chose this voice, so the first clip stands for it. The voice id names the synthesiser, its voice
        # and its version, so every run with the same DONOR_MODEL has the same voice id and train_vits can combine them.
        shutil.copyfile(out_dir / FINAL_ROWS[0]["audio"], VOICE_REF)
        VOICE_REF_TEXT = FINAL_ROWS[0]["text"]
        (out_dir / "voice_ref.txt").write_text(VOICE_REF_TEXT + "\n", encoding="utf-8")
        VOICE_ID = hashlib.sha1(f"{DONOR_PLUGIN_ID}|{DONOR_VOICE_ID}|{DONOR_VERSION}".encode()).hexdigest()[:12]
    else:
        VOICE_ID = hashlib.sha1(VOICE_REF.read_bytes()).hexdigest()[:12]
        DONOR_VOICE_ID = f"{voice_mode}:{VOICE_ID}"

    manifest_rows = []
    for row in FINAL_ROWS:
        manifest_rows.append({
            "id": row["id"],
            "text": row["text"],
            "lang": TTS_LANG,
            "audio": row["audio"],
            "duration_s": row["duration_s"],
            "sample_rate": row["sample_rate"],
            "source": {
                "kind": "tts",
                "plugin": DONOR_PLUGIN_ID,
                "voice": DONOR_VOICE_ID,
                "version": DONOR_VERSION,
            },
        })

    manifest_path = out_dir / MANIFEST_NAME
    with open(manifest_path, "w", encoding="utf-8") as f:
        for row in manifest_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    metadata_csv_path = out_dir / "metadata.csv"
    with open(metadata_csv_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f, delimiter="|")          # a text with a quote in it is quoted, the way Python's csv module reads it
        for row in manifest_rows:
            writer.writerow([Path(row["audio"]).name, row["text"]])

    total_seconds = round(sum(r["duration_s"] for r in manifest_rows), 2)
    prompts_source = {
        "kind": prompts_source_kind,
        "name": prompts_source_name,
        "path": (prompts_source_path if prompts_source_kind == "tatoeba" else os.path.basename(prompts_source_path)) if prompts_source_path else None,
        "licence": PROMPT_LICENCE,
        "shard": SHARD if shard_applies else None,
        "shards": PROMPT_SHARDS if shard_applies else None,
    }
    donor = {
        "plugin": DONOR_PLUGIN_ID,
        "voice": DONOR_VOICE_ID,
        "version": DONOR_VERSION,
        "mode": DONOR_MODE,
        "ci_path_only": DONOR_MODE == "ci",
        "note": DONOR_NOTE,
        **DONOR_DETAILS,
    }
    dataset = {
        "tts_dataset": DATASET_FORMAT,
        "generator": "ngi0-notebooks/tts_dataset_gen",
        "run_id": RUN_ID,
        "language": TTS_LANG,
        "sample_rate": TARGET_SR,
        "clips": len(manifest_rows),
        "total_seconds": total_seconds,
        "voice_id": VOICE_ID,
        "ref_text": VOICE_REF_TEXT,
        "seed": SEED,
        "donor": donor,
        "prompts_source": prompts_source,
    }
    with open(out_dir / "dataset.json", "w", encoding="utf-8") as f:
        json.dump(dataset, f, ensure_ascii=False, indent=2)

    meta = {
        "notebooks": ["tts_dataset_gen"],
        "run_id": RUN_ID,
        "voice_id": VOICE_ID,
        "prompts_source": prompts_source,
        "donor": donor,
        "generation": GENERATION,
        "counts": {
            "prompt_lines_read": len(raw_lines),
            "prompts_refused": {k: v for k, v in reason_counts.items() if k != "passed"},
            "prompts_passed_text_validation": reason_counts.get("passed", 0),
            "donor_synthesis_failed": n_synth_failed,
            "post_processing_failed": n_post_failed,
            "dropped_for_length": n_bad_length,
            "rows_written": len(manifest_rows),
            "hours_written": round(total_seconds / 3600, 4),
        },
    }
    meta_path = out_dir / (MANIFEST_NAME + ".meta.json")
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    print(f"Wrote {len(manifest_rows)} clips, {total_seconds / 60:.1f} minutes, voice {VOICE_ID}, to {out_dir}")
    tick("write the dataset")

In [ ]:
# Helper code, hidden by default: reads the folder back and stops if anything in it is wrong. Nothing here needs editing.
def check_dataset():
    # every row has its clip, lengths are in bounds, one voice, no empty text
    with open(out_dir / "metadata.csv", encoding="utf-8", newline="") as f:
        meta_rows = [row for row in csv.reader(f, delimiter="|") if row]
    ds = json.loads((out_dir / "dataset.json").read_text(encoding="utf-8"))
    with open(out_dir / MANIFEST_NAME, encoding="utf-8") as f:
        man_rows = [json.loads(line) for line in f]

    assert meta_rows, "metadata.csv is empty"
    assert all(len(r) == 2 for r in meta_rows), "a metadata.csv row is not one file name and one text, separated by '|'"
    assert all(text.strip() for _, text in meta_rows), "a metadata.csv row has no text"
    names = [name for name, _ in meta_rows]
    assert len(set(names)) == len(names), "a file name is in metadata.csv twice"
    assert all(f"r{RUN_ID}-" in name for name in names), "a file name does not carry the run id"
    missing = [name for name in names if not (out_dir / "wavs" / name).is_file()]
    assert not missing, f"{len(missing)} metadata.csv rows have no audio file, for example {missing[0]}"
    assert {p.name for p in (out_dir / "wavs").iterdir()} == set(names), "wavs/ holds a file that metadata.csv does not list"
    infos = [sf.info(str(out_dir / "wavs" / name)) for name in names]
    assert all(i.samplerate == TARGET_SR and i.channels == 1 for i in infos), f"a clip is not mono at {TARGET_SR} Hz"
    durations = [i.frames / i.samplerate for i in infos]
    assert MIN_SECONDS <= min(durations) and max(durations) <= MAX_SECONDS, (
        f"clip lengths run from {min(durations):.2f} to {max(durations):.2f} s, outside {MIN_SECONDS} to {MAX_SECONDS} s")
    assert ds["tts_dataset"] == DATASET_FORMAT and ds["run_id"] == RUN_ID and ds["sample_rate"] == TARGET_SR, "dataset.json does not describe this run"
    assert ds["clips"] == len(meta_rows) == len(man_rows), "dataset.json, metadata.csv and the manifest count different clips"
    assert abs(ds["total_seconds"] - sum(durations)) < 1 + 0.001 * len(durations), "dataset.json's total_seconds is not the length of the clips"
    assert ds["ref_text"].strip() and ds["ref_text"].strip() == (out_dir / "voice_ref.txt").read_text(encoding="utf-8").strip(), "ref_text is not the content of voice_ref.txt"
    if DONOR_MODE == "omnivoice":
        assert ds["voice_id"] == hashlib.sha1((out_dir / "voice_ref.wav").read_bytes()).hexdigest()[:12], "voice_id is not the SHA-1 of voice_ref.wav"
    else:
        assert ds["voice_id"] == hashlib.sha1(f"{donor['plugin']}|{donor['voice']}|{donor['version']}".encode()).hexdigest()[:12], \
            "voice_id is not the SHA-1 of the donor's plugin, voice and version"
    if SMOKE:
        # the next run finds this voice when REF_AUDIO names a folder that holds this one
        _parent = Path(WORK_DIR) / "smoke_ref_audio"
        shutil.rmtree(_parent, ignore_errors=True)
        (_parent / out_dir.name).mkdir(parents=True)
        for _name in ("voice_ref.wav", "voice_ref.txt"):
            shutil.copyfile(out_dir / _name, _parent / out_dir.name / _name)
        assert voice_in_folder(_parent) == (str(_parent / out_dir.name / "voice_ref.wav"), ds["ref_text"].strip()), \
            "REF_AUDIO cannot find the voice in a folder that holds this run"
    voices = {r["source"]["voice"] for r in man_rows}
    assert len(voices) == 1, f"the manifest names {len(voices)} voices; a dataset has one"
    if DONOR_MODE == "omnivoice":
        assert voices == {f"{voice_mode}:{ds['voice_id']}"}, "the manifest's voice is not the voice of dataset.json"
    assert [(Path(r["audio"]).name, r["text"]) for r in man_rows] == [tuple(r) for r in meta_rows], "the manifest and metadata.csv differ"

    size_mb = sum(p.stat().st_size for p in [*out_dir.iterdir(), *(out_dir / "wavs").iterdir()] if p.is_file()) / 1e6
    words = [len(text.split()) for _, text in meta_rows]
    print("=" * 72)
    print(f"Run {RUN_ID}, language {TTS_LANG}, seed {SEED}" + ("  [SMOKE test: espeak-ng]" if SMOKE else ""))
    print(f"Clips     : {len(meta_rows)} of {len(PROMPT_ROWS)} sentences; {n_synth_failed} not spoken, {n_post_failed} dropped")
    print(f"Speech    : {sum(durations) / 60:.1f} minutes; a clip is {min(durations):.1f} to {max(durations):.1f} s, {np.mean(durations):.1f} s on average, "
          f"{np.mean(words):.1f} words on average")
    print(f"Voice     : {ds['voice_id']} ({donor['note']})")
    print(f"Sentences : {prompts_source['name']}" + (f", group {SHARD} of {PROMPT_SHARDS}" if shard_applies else ""))
    if GENERATION.get("clips_per_s"):
        print(f"Generation: {GENERATION['clips']} clips in {GENERATION['seconds'] / 60:.1f} min = {GENERATION['clips_per_s']} clips/s"
              + (f", {GENERATION['audio_s_per_s']} s of speech per second; GPU memory peak {GENERATION['gpu_memory']['peak_allocated_gib']} GiB "
                 f"of {GENERATION['gpu_memory']['total_gib']} GiB" if "gpu_memory" in GENERATION else ""))
    print(f"Folder    : {size_mb:.0f} MB")
    tick("check")
    print("Time per step:")
    for _name, _sec in _STEPS.items():
        print(f"  {_name:24s} {_sec / 60:6.1f} min")
    print(f"  {'total':24s} {sum(_STEPS.values()) / 60:6.1f} min")
    print("A few clips:")
    for _name, _text in meta_rows[:: max(1, len(meta_rows) // 5)][:5]:
        print(f"  {_name}  {_text}")
    print("=" * 72)

In [ ]:
if SR_ENGINE:
    %pip install --quiet --pre "audiosronnx>=0.9.0a2" "onnxruntime>=1.30.0"
choose_sentences()
speak_sentences()
clean_clips()
write_dataset()
check_dataset()

## 4. What you got

| In the folder | What it is |
|---|---|
| `wavs/` | the audio clips |
| `metadata.csv` | one line per clip: `<audio filename>|<text>`; a text with a `"` in it is put in quotes, as in any CSV file |
| `dataset.json` | the facts of the run: language, clips, seconds, `voice_id` |
| `voice_ref.wav`, `voice_ref.txt` | the voice, and the words spoken in it |
| `manifest.jsonl` | the same clips for the later notebooks of [docs/pipeline.md](../docs/pipeline.md) |

The sentences are from Tatoeba, under the CC BY 2.0 FR licence. Name Tatoeba if you publish the dataset.

Tatoeba sentences are not filtered, and some contain swearing or violence. Tatoeba does not mark language varieties either: `pt-PT` draws European and Brazilian Portuguese alike, and `zh` mixes simplified and traditional Chinese characters.

In [ ]:
print(f"Dataset ready: {out_dir}")
for name in ("wavs/", "metadata.csv", "dataset.json", "voice_ref.wav", "voice_ref.txt", MANIFEST_NAME, MANIFEST_NAME + ".meta.json"):
    p = out_dir / name
    if p.is_dir():
        print(f"  {name:28s} {len(list(p.glob('*.wav')))} WAV files")
    elif p.suffix in (".csv", ".jsonl"):
        print(f"  {name:28s} {sum(1 for _ in open(p, encoding='utf-8'))} rows")
    else:
        print(f"  {name:28s} {p.stat().st_size} bytes")
shutil.rmtree(WORK_DIR, ignore_errors=True)            # the donor's raw audio; the dataset is kept

## 5. Train a voice, or make more data first

**Train.** Open [`train_vits.ipynb`](train_vits.ipynb) and set its `DATASET_DIRS` to this folder. On Kaggle: Add Input, Notebook output, pick this run.

**More data.** One run makes about 75 minutes of speech. For more, make several runs, at the same time if you like, then train once from all the folders:

1. Run 1 as it is.
2. In every other run set `RUN_ID` to `"2"`, `"3"` and so on, and `REF_AUDIO` to run 1's folder. On Kaggle: Add Input, Notebook output, pick run 1, and set `REF_AUDIO = "/kaggle/input"`.

Each `RUN_ID` from 1 to 16 speaks its own Tatoeba sentences, and all runs share run 1's voice: the notebook finds run 1's recording `voice_ref.wav` and its words in that folder, and copies the voice from it. Do not let every run invent the voice again, because two runs invent the same voice only on the same GPU model with the same software. With `DONOR_MODEL`, leave `REF_AUDIO` empty: every run with the same `DONOR_MODEL` and the same package version has the same voice. Start parallel `DONOR_MODEL` runs close together, because a run that installs a newer version of the plugin gets another voice, and `train_vits` refuses to mix them.

**Your own sentences.** Set `PROMPTS_FILE` in the advanced settings to a text file with one sentence per line. Every run speaks the whole file, unless you also give each run its own `PROMPT_SHARD` (`"0"` to `"15"`): then each run speaks only its own part of the file.

### If something goes wrong

| What you see | What to do |
|---|---|
| `OmniVoice needs a GPU` | Kaggle: Settings panel, Accelerator, GPU T4, then Run All again. |
| `Could not read https://downloads.tatoeba.org/...` | Switch Internet on (Kaggle: Settings panel, Internet), then Run All again. |
| `Tatoeba has only ... usable sentences` | Lower `N_PROMPTS`, or set `PROMPTS_FILE` to your own sentences. |
| `OmniVoice does not support TTS_LANG` | Set `DONOR_MODEL` in the advanced settings to a phoonnx voice or an OVOS TTS plugin. |
| `RUN_ID=... must be a whole number` | Give each run its own number from `"1"` to `"16"`. |
| `REF_AUDIO needs REF_TEXT` | Type the exact words of the recording into `REF_TEXT`. |
| `REF_AUDIO=... does not exist` | Check the path. On Kaggle, files you add are under `/kaggle/input`. |
| The session timed out | Lower `N_PROMPTS` and press Run All again. |

[![NGI0 Commons Fund](../ngi.png)](https://nlnet.nl/project/OpenVoiceOS)

This project was funded through the [NGI0 Commons Fund](https://nlnet.nl/commonsfund), a fund established by [NLnet](https://nlnet.nl) with financial support from the European Commission's [Next Generation Internet](https://ngi.eu) programme, under the aegis of DG Communications Networks, Content and Technology under grant agreement No [101135429](https://cordis.europa.eu/project/id/101135429).

> This notebook was written with AI assistance (Claude, via Claude Code). Verify outputs before relying on them.

Licensed under Apache-2.0.